# Lockbox State Prediction

Predict the discrete lockbox **stage** (0 to 4) plus the four continuous mechanism
states from six feature variants × three model architectures.

**Inputs**
- `lockbox_state.pkl` (Notebook 06), ground-truth stage + continuous mech states + lockbox 2D/3D
- `pipeline_state.pkl`, mouse 2D + mouse 3D after all fallbacks (≈ 58.6 % coverage)
- `points_3d_all_sources.csv`, mouse 3D with NN inpainting (`source == 'nn'`)

**Feature variants**

| name | source | per-frame dim |
|---|---|---|
| LK-2D | lockbox 2D × 3 views (x, y, likelihood per kp) | `3 · n_lk · 3` |
| LK-3D | lockbox 3D triangulated (x, y, z, mask) | `n_lk · 4` |
| MS-2D | mouse 2D × 3 views | `3 · n_ms · 3` |
| MS-3D-raw | mouse 3D after fallbacks (no NN) | `n_ms · 4` |
| MS-3D-nn | mouse 3D after NN inpainting | `n_ms · 4` |
| LK+MS-3D-nn | concat of LK-3D and MS-3D-nn (ceiling check) | combined |

**Targets**
- Primary: `stage` ∈ {0..4} as regression (monotone-by-construction).
- Auxiliary: `[lever_rad, slider_mm, ball_mm, cover_mm]` as 4-dim regression, masked where ground truth is NaN.

The lever's *individual* state isn't strictly monotonic (it can flip back), but the discrete
`stage` is, so we apply monotone-non-decreasing post-processing **only** to the stage prediction
when extracting onsets, never to the per-mechanism aux head.

**Models**
- **XGBoost**, per-frame baseline, no temporal context, NaN-native.
- **1D-CNN**, three Conv1d blocks over a window of `±15` frames + global pool.
- **Transformer**, 3-layer encoder, 4 heads, d_model = 64, prediction from the centre token.

**Eval**
- 5-fold block CV (50-frame blocks, interleaved) + a final-10 % strip held out from *every* fold.
- Metrics: `stage_mae`, `stage_acc` (rounded), `onset_err_s`, per-mechanism MAE.
- Plots: predicted stage timelines for every (model × feature-set), MAE bars.

## 0. Imports, paths, hyper-params

In [1]:
import pickle, json, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import Dataset, DataLoader
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}  on {DEVICE}')

torch 2.3.1+cu121  on cuda


In [ ]:
# Paths (adjust to your layout)
LOCKBOX_STATE_PKL  = Path(r'../data/lockbox_state/scene1/lockbox_state.pkl').resolve()
PIPELINE_STATE_PKL = Path('../data/triangulate_render/scene1/pipeline_state.pkl').resolve()
NN_INPAINT_CSV     = Path('../data/nn_output/scene1/points_3d_all_sources.csv').resolve()
OUT_DIR            = Path(r'../data/lockbox_state_prediction/scene1').resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Hyper-params (one place to tweak)
FPS          = 30
WINDOW       = 31     # ± 15 frames around target = 1 s of context at 30 fps
BLOCK        = 50     # CV block size in frames
N_FOLDS      = 5
HOLDOUT_FRAC = 0.10   # final strip held out from every fold

# NN training
EPOCHS_NN    = 15
BATCH        = 128
LR           = 1e-3
AUX_WEIGHT   = 0.10   # weight of the aux mechanism loss vs the stage loss

# XGBoost
XGB_ESTIMATORS = 400
XGB_DEPTH      = 6
XGB_LR         = 0.05

HALF = WINDOW // 2

## 1. Ground truth from `lockbox_state.pkl`

`stage` is monotonic by construction. `aux_target` carries the four per-mechanism
continuous values (lever rad, slider mm, ball-distance mm, cover mm) as auxiliary
regression targets.

In [3]:
with open(LOCKBOX_STATE_PKL, 'rb') as f:
    lk = pickle.load(f)

stage       = lk['stage'].astype(np.float32)             # (n_frames,)
n_frames    = int(lk['n_frames'])
STAGE_NAMES = lk['STAGE_NAMES']
STAGE_ORDER = lk['STAGE_ORDER']                          # ['lever1','slider1','ball1','cover1']

# 4-dim auxiliary target (continuous per-mechanism state)
aux_target = np.column_stack([lk['state'][m]['value'] for m in STAGE_ORDER]).astype(np.float32)
aux_units  = [lk['state'][m]['unit'] for m in STAGE_ORDER]

# Raw lockbox arrays for LK-2D / LK-3D
lk_2d_xy = lk['points_2d']        # (3, n_frames, n_lk, 2), NaN where likelihood < CONF
lk_lik   = lk['likelihoods']      # (3, n_frames, n_lk), always finite
lk_3d    = lk['points_3d']        # (n_frames, n_lk, 3), NaN where not reconstructed
lk_valid = lk['valid_kps']
lk_view  = lk['view_order']
n_lk     = len(lk_valid)

print(f'n_frames           : {n_frames}')
print(f'lk views/kps       : {lk_view}  /  {n_lk} kps')
print(f'stage histogram    : ' + ', '.join(
    f'{STAGE_NAMES[s]}={int((stage==s).sum())}' for s in range(len(STAGE_NAMES))))
print(f'aux mechs (units)  : {list(zip(STAGE_ORDER, aux_units))}')
print(f'aux finite per kp  : {(np.isfinite(aux_target).mean(axis=0)*100).round(1)} %')

n_frames           : 8190
lk views/kps       : ['top', 'side', 'front']  /  8 kps
stage histogram    : start=5393, lever1_pivoted=207, slider1_slid=665, ball_removed=78, cover_slid=1847
aux mechs (units)  : [('lever1', 'rad'), ('slider1', 'mm'), ('ball1', 'mm'), ('cover1', 'mm')]
aux finite per kp  : [87.6 98.  81.6 80.5] %


## 2. Build the six feature variants

Every variant becomes a per-frame 1-D vector.

- **2-D**: concatenated `(x, y, likelihood)` per camera per keypoint. We use the raw
  likelihood as confidence, no thresholding.
- **3-D**: `(x, y, z, finite-mask)` per keypoint. NaN values are zero-filled *after* the
  mask is computed, so the model sees "feature missing to mask=0 ∧ value=0".

In [4]:
def build_2d_features(xy, lik):
    """xy: (n_cams, n_frames, n_kps, 2) (NaN allowed); lik: (n_cams, n_frames, n_kps)."""
    n_c, n_f, n_k, _ = xy.shape
    out = np.zeros((n_f, n_c * n_k * 3), dtype=np.float32)
    for c in range(n_c):
        for k in range(n_k):
            b = (c * n_k + k) * 3
            out[:, b+0] = np.nan_to_num(xy[c, :, k, 0], nan=0.0)
            out[:, b+1] = np.nan_to_num(xy[c, :, k, 1], nan=0.0)
            out[:, b+2] = np.nan_to_num(lik[c, :, k].astype(np.float32), nan=0.0)
    return out

def build_3d_features(p3d):
    """p3d: (n_frames, n_kps, 3) with NaN where unreconstructed."""
    n_f, n_k, _ = p3d.shape
    out = np.zeros((n_f, n_k * 4), dtype=np.float32)
    for k in range(n_k):
        b = k * 4
        out[:, b:b+3] = np.nan_to_num(p3d[:, k, :], nan=0.0)
        out[:, b+3]   = np.isfinite(p3d[:, k, :]).all(axis=-1).astype(np.float32)
    return out

# ── Lockbox features ──────────────────────────────────────────────────
F_LK_2D = build_2d_features(lk_2d_xy, lk_lik)
F_LK_3D = build_3d_features(lk_3d)
print(f'LK-2D : {F_LK_2D.shape}')
print(f'LK-3D : {F_LK_3D.shape}')

LK-2D : (8190, 72)
LK-3D : (8190, 32)


### Mouse features, load `pipeline_state.pkl`

In [5]:
with open(PIPELINE_STATE_PKL, 'rb') as f:
    ms = pickle.load(f)

ms_3d_raw    = ms['points_3d']                  # (n_frames_m, n_ms, 3) post-fallback (≈58.6%)
ms_valid     = ms['valid_kps']
ms_views_3d  = ms['VIEWS_3D']                   # views the mouse pipeline used
ms_view_pred = ms['view_predictions']           # dict: view -> DLC DataFrame
n_ms         = len(ms_valid)

# Align frame counts in case of a slight mismatch between pipelines
n_frames_use = min(n_frames, ms_3d_raw.shape[0])
if n_frames_use < n_frames:
    print(f'⚠ truncating to {n_frames_use} (lockbox had {n_frames}, mouse has {ms_3d_raw.shape[0]})')
    stage      = stage[:n_frames_use]
    aux_target = aux_target[:n_frames_use]
    F_LK_2D    = F_LK_2D[:n_frames_use]
    F_LK_3D    = F_LK_3D[:n_frames_use]
ms_3d_raw = ms_3d_raw[:n_frames_use]
n_frames  = n_frames_use

# Build mouse 2D feature tensor by reading view_predictions DataFrames
ms_2d_xy = np.full((len(ms_views_3d), n_frames, n_ms, 2), np.nan, dtype=np.float32)
ms_lik   = np.zeros((len(ms_views_3d), n_frames, n_ms), dtype=np.float32)
for c, v in enumerate(ms_views_3d):
    if v not in ms_view_pred:
        print(f'⚠ mouse view {v!r} missing from view_predictions; left as NaN')
        continue
    df = ms_view_pred[v]
    while df.columns.nlevels > 2:
        df = df.droplevel(0, axis=1)
    present = set(df.columns.get_level_values(0))
    for k, bp in enumerate(ms_valid):
        if bp not in present:
            continue
        ms_2d_xy[c, :, k, 0] = df[bp]['x'].values[:n_frames]
        ms_2d_xy[c, :, k, 1] = df[bp]['y'].values[:n_frames]
        ms_lik  [c, :, k]    = df[bp]['likelihood'].values[:n_frames].astype(np.float32)

F_MS_2D     = build_2d_features(ms_2d_xy, ms_lik)
F_MS_3D_raw = build_3d_features(ms_3d_raw)

cov_ms = np.isfinite(ms_3d_raw).all(axis=-1).mean() * 100
print(f'mouse kps                          : {n_ms}')
print(f'mouse 3D coverage (post-fallback)  : {cov_ms:.1f} %')
print(f'MS-2D     : {F_MS_2D.shape}')
print(f'MS-3D-raw : {F_MS_3D_raw.shape}')

mouse kps                          : 20
mouse 3D coverage (post-fallback)  : 58.6 %
MS-2D     : (8190, 180)
MS-3D-raw : (8190, 80)


### Mouse features, NN-inpainted 3D

In [6]:
ms_3d_nn = np.full((n_frames, n_ms, 3), np.nan, dtype=np.float32)
if NN_INPAINT_CSV.exists():
    df_nn = pd.read_csv(NN_INPAINT_CSV)
    df_nn = df_nn[df_nn['source'] == 'nn']
    kp_to_idx = {bp: i for i, bp in enumerate(ms_valid)}
    skipped = 0
    for r in df_nn.itertuples(index=False):
        f = int(r.frame); k = kp_to_idx.get(r.keypoint)
        if k is None or f >= n_frames:
            skipped += 1; continue
        ms_3d_nn[f, k] = (r.x, r.y, r.z)
    cov_nn = np.isfinite(ms_3d_nn).all(axis=-1).mean() * 100
    print(f'mouse 3D coverage (NN inpainted)   : {cov_nn:.1f} %   (skipped {skipped} rows)')
else:
    print(f'⚠ {NN_INPAINT_CSV} not found — using MS-3D-raw as MS-3D-nn placeholder.')
    ms_3d_nn = ms_3d_raw.copy()

F_MS_3D_nn = build_3d_features(ms_3d_nn)
print(f'MS-3D-nn  : {F_MS_3D_nn.shape}')

mouse 3D coverage (NN inpainted)   : 100.0 %   (skipped 0 rows)
MS-3D-nn  : (8190, 80)


### Combine + standardize all feature sets

In [ ]:
F_LKMS = np.concatenate([F_LK_3D, F_MS_3D_nn], axis=1)  # ceiling check

FEATURE_SETS = {
    'LK-2D'       : F_LK_2D,
    'LK-3D'       : F_LK_3D,
    'MS-2D'       : F_MS_2D,
    'MS-3D-raw'   : F_MS_3D_raw,
    'MS-3D-nn'    : F_MS_3D_nn,
    'LK+MS-3D-nn' : F_LKMS,
}

# Standardize globally for inspection
FEATURE_SETS_NORM = {}
for name, feats in FEATURE_SETS.items():
    mu = feats.mean(0); sd = feats.std(0); sd[sd < 1e-6] = 1.0
    FEATURE_SETS_NORM[name] = ((feats - mu) / sd).astype(np.float32)

print('Feature inventory:')
for name, feats in FEATURE_SETS.items():
    print(f'  {name:<14} dim={feats.shape[1]:>4}')

Feature inventory:
  LK-2D          dim=  72
  LK-3D          dim=  32
  MS-2D          dim= 180
  MS-3D-raw      dim=  80
  MS-3D-nn       dim=  80
  LK+MS-3D-nn    dim= 112


## 3. CV splits

50-frame blocks shuffled into 5 folds. The final 10 % of frames are held out from
every fold and used as a "future-time" check. For windowed NN training we let the
dataset filter out frames within `HALF` of the sequence ends. We don't apply an
extra boundary buffer here because the blocks are small enough (50 frames) that
the contribution from cross-block window overlap is bounded.

In [8]:
def make_cv_splits(n_frames, block=BLOCK, n_folds=N_FOLDS, holdout_frac=HOLDOUT_FRAC, seed=SEED):
    holdout_start = int(n_frames * (1 - holdout_frac))
    holdout_mask  = np.zeros(n_frames, dtype=bool); holdout_mask[holdout_start:] = True
    cv_end        = holdout_start

    n_blocks = cv_end // block
    rng = np.random.default_rng(seed)
    block_ids = rng.permutation(n_blocks)
    fold_of   = np.full(n_blocks, -1, int)
    for i, b in enumerate(block_ids):
        fold_of[b] = i % n_folds

    folds = []
    for f in range(n_folds):
        tr = np.zeros(n_frames, dtype=bool)
        te = np.zeros(n_frames, dtype=bool)
        for b in range(n_blocks):
            s, e = b * block, (b + 1) * block
            (te if fold_of[b] == f else tr)[s:e] = True
        folds.append((tr, te))
    return folds, holdout_mask

folds, holdout_mask = make_cv_splits(n_frames)
print(f'Holdout strip   : frames {int(holdout_mask.argmax())}..{n_frames-1}  ({holdout_mask.sum()})')
for i, (tr, te) in enumerate(folds):
    print(f'  fold {i}: train={tr.sum():>5}   test={te.sum():>5}')

Holdout strip   : frames 7371..8189  (819)
  fold 0: train= 5850   test= 1500
  fold 1: train= 5850   test= 1500
  fold 2: train= 5900   test= 1450
  fold 3: train= 5900   test= 1450
  fold 4: train= 5900   test= 1450


## 4. Metrics

- `stage_mae`: MAE on the raw stage prediction (clipped to [0, 4])
- `stage_acc`: rounded-prediction classification accuracy across stage classes
- `onset_err_s`: mean absolute onset error in seconds, computed only over stages the model reaches
- `mae_<mech>`: per-mechanism MAE in native units, only over frames with finite ground truth

In [9]:
def find_onsets(stage_curve, n_stages=None):
    """Monotone-non-decreasing post-processing, then threshold at 0.5, 1.5, …
    Returns dict {stage_idx (1..n_stages-1) -> frame or None}."""
    n_stages = n_stages or len(STAGE_NAMES)
    s = np.maximum.accumulate(np.nan_to_num(stage_curve, nan=-np.inf))
    out = {}
    for k in range(1, n_stages):
        hit = np.where(s >= k - 0.5)[0]
        out[k] = int(hit[0]) if len(hit) else None
    return out

def compute_metrics(stage_pred, stage_true, mech_pred=None, mech_true=None, fps=FPS):
    valid = np.isfinite(stage_pred) & np.isfinite(stage_true)
    if valid.sum() < 2:
        return dict(stage_mae=np.nan, stage_acc=np.nan, onset_err_s=np.nan)
    sp = np.clip(stage_pred[valid], 0, len(STAGE_NAMES) - 1)
    st = stage_true[valid].astype(int)
    out = dict(
        stage_mae = float(np.mean(np.abs(sp - st))),
        stage_acc = float(np.mean(np.round(sp).astype(int) == st)),
    )

    # Onset error: predict over the full curve (with NaNs left as -inf for cummax)
    po = find_onsets(stage_pred)
    to = find_onsets(stage_true)
    errs = [abs(po[k] - to[k]) / fps for k in po
            if po[k] is not None and to[k] is not None]
    out['onset_err_s'] = float(np.mean(errs)) if errs else float('nan')

    if mech_pred is not None and mech_true is not None:
        for i, m in enumerate(STAGE_ORDER):
            f = np.isfinite(mech_true[:, i]) & np.isfinite(mech_pred[:, i])
            out[f'mae_{m}'] = float(np.mean(np.abs(mech_pred[f, i] - mech_true[f, i]))) if f.any() else float('nan')
    return out

## 5. XGBoost (per-frame baseline)

One regressor per target per fold. Five mech / stage regressors × five folds × six
feature sets is fast. Tens of seconds total.

In [10]:
def xgb_fit_predict(feats, y_train_full, train_idx, test_idx):
    """Fit XGBRegressor on (feats[train_idx], y_train_full[train_idx]) where target is finite."""
    finite = np.isfinite(y_train_full[train_idx])
    if finite.sum() < 5:
        return np.full(len(test_idx), np.nan, dtype=np.float32)
    model = XGBRegressor(
        n_estimators=XGB_ESTIMATORS, max_depth=XGB_DEPTH, learning_rate=XGB_LR,
        n_jobs=-1, random_state=SEED, verbosity=0, tree_method='hist',
    )
    model.fit(feats[train_idx][finite], y_train_full[train_idx][finite])
    return model.predict(feats[test_idx]).astype(np.float32)

xgb_results = {}
t0 = time.time()
print('Training XGBoost per feature set (5-fold CV + holdout)…\n')
for name, feats in FEATURE_SETS_NORM.items():
    stage_pred = np.full(n_frames, np.nan, dtype=np.float32)
    mech_pred  = np.full((n_frames, 4), np.nan, dtype=np.float32)

    for tr_m, te_m in folds:
        tr_idx, te_idx = np.where(tr_m)[0], np.where(te_m)[0]
        stage_pred[te_idx] = xgb_fit_predict(feats, stage, tr_idx, te_idx)
        for i in range(4):
            mech_pred[te_idx, i] = xgb_fit_predict(feats, aux_target[:, i], tr_idx, te_idx)

    cv_idx, ho_idx = np.where(~holdout_mask)[0], np.where(holdout_mask)[0]
    stage_ho = xgb_fit_predict(feats, stage, cv_idx, ho_idx)
    mech_ho  = np.column_stack([xgb_fit_predict(feats, aux_target[:, i], cv_idx, ho_idx) for i in range(4)])

    full_stage = stage_pred.copy(); full_stage[ho_idx] = stage_ho
    full_mech  = mech_pred.copy();  full_mech [ho_idx] = mech_ho

    cv_eval = np.zeros(n_frames, dtype=bool)
    for _, te in folds: cv_eval |= te
    m_cv = compute_metrics(full_stage[cv_eval], stage[cv_eval], full_mech[cv_eval], aux_target[cv_eval])
    m_ho = compute_metrics(full_stage[ho_idx], stage[ho_idx], full_mech[ho_idx], aux_target[ho_idx])

    xgb_results[name] = dict(cv=m_cv, holdout=m_ho, stage_pred=full_stage, mech_pred=full_mech)
    print(f'  {name:<14}  CV mae={m_cv["stage_mae"]:.3f} acc={m_cv["stage_acc"]:.3f}   '
          f'HO mae={m_ho["stage_mae"]:.3f} acc={m_ho["stage_acc"]:.3f}')
print(f'\n[xgb] done in {time.time()-t0:.1f}s')

Training XGBoost per feature set (5-fold CV + holdout)…



  LK-2D           CV mae=0.008 acc=0.991   HO mae=0.314 acc=0.724


  LK-3D           CV mae=0.004 acc=0.996   HO mae=0.150 acc=0.932


  MS-2D           CV mae=0.219 acc=0.873   HO mae=3.016 acc=0.159


  MS-3D-raw       CV mae=0.228 acc=0.853   HO mae=2.999 acc=0.158


  MS-3D-nn        CV mae=0.203 acc=0.872   HO mae=2.887 acc=0.164


  LK+MS-3D-nn     CV mae=0.004 acc=0.996   HO mae=0.150 acc=0.932

[xgb] done in 199.6s


## 6. Windowed NN models (CNN + Transformer)

Same dataset and training loop for both. The Dataset returns a (`WINDOW × D`) tensor
of standardized features and the frame index, so we can re-assemble per-frame
predictions from minibatched eval output.

In [11]:
class WindowDataset(Dataset):
    """Yields (window_features, stage_target, mech_target_z, mech_mask, frame_idx)."""
    def __init__(self, feats, stage_t, mech_t_z, frame_indices):
        self.feats = feats; self.stage = stage_t; self.mech = mech_t_z
        n = len(feats)
        idx = np.asarray(frame_indices)
        ok = (idx >= HALF) & (idx < n - HALF)
        self.idx = idx[ok].astype(np.int64)
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        f = int(self.idx[i])
        x = self.feats[f - HALF : f + HALF + 1]
        ym  = self.mech[f]
        mm  = np.isfinite(ym)
        return (
            torch.from_numpy(x),
            torch.tensor(self.stage[f], dtype=torch.float32),
            torch.from_numpy(np.nan_to_num(ym, nan=0.0)).float(),
            torch.from_numpy(mm).bool(),
            torch.tensor(f, dtype=torch.long),
        )

class TempCNN(nn.Module):
    def __init__(self, in_dim, hidden=128):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv1d(in_dim, 64, 5, padding=2), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, hidden, 5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Conv1d(hidden, hidden, 5, padding=2), nn.BatchNorm1d(hidden), nn.ReLU(),
        )
        self.head_s = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 1))
        self.head_m = nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Linear(64, 4))
    def forward(self, x):                                    # x: (B, W, D)
        z = self.body(x.transpose(1, 2)).mean(dim=2)         # (B, hidden)
        return self.head_s(z).squeeze(-1), self.head_m(z)

class TempTransformer(nn.Module):
    def __init__(self, in_dim, d_model=64, nhead=4, nlayers=3):
        super().__init__()
        self.proj = nn.Linear(in_dim, d_model)
        self.pos  = nn.Parameter(torch.randn(WINDOW, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=128, dropout=0.1,
            batch_first=True, activation='gelu',
        )
        self.tr = nn.TransformerEncoder(enc, num_layers=nlayers)
        self.head_s = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 1))
        self.head_m = nn.Sequential(nn.Linear(d_model, 64), nn.ReLU(), nn.Linear(64, 4))
    def forward(self, x):
        z = self.proj(x) + self.pos.unsqueeze(0)             # (B, W, d_model)
        z = self.tr(z)
        c = z[:, HALF]                                       # centre token
        return self.head_s(c).squeeze(-1), self.head_m(c)

In [12]:
def train_one_split(model_cls, feats, stage_t, mech_t_z, tr_idx, te_idx,
                     epochs=EPOCHS_NN, batch=BATCH, lr=LR, aux_w=AUX_WEIGHT):
    in_dim = feats.shape[1]
    model  = model_cls(in_dim).to(DEVICE)
    opt    = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    ds_tr = WindowDataset(feats, stage_t, mech_t_z, tr_idx)
    ds_te = WindowDataset(feats, stage_t, mech_t_z, te_idx)
    if len(ds_tr) == 0 or len(ds_te) == 0:
        return np.array([], int), np.array([]), np.array([]).reshape(0, 4)

    loader_tr = DataLoader(ds_tr, batch_size=batch, shuffle=True,  num_workers=0)
    loader_te = DataLoader(ds_te, batch_size=batch, shuffle=False, num_workers=0)

    for _ in range(epochs):
        model.train()
        for x, ys, ym, mm, _ in loader_tr:
            x, ys, ym, mm = x.to(DEVICE), ys.to(DEVICE), ym.to(DEVICE), mm.to(DEVICE)
            ps, pm = model(x)
            loss_s = Fnn.mse_loss(ps, ys)
            diff_m = (pm - ym).pow(2) * mm.float()
            loss_m = diff_m.sum() / mm.float().sum().clamp_min(1.0)
            loss   = loss_s + aux_w * loss_m
            opt.zero_grad(); loss.backward(); opt.step()

    # Eval
    model.eval()
    s_all, m_all, f_all = [], [], []
    with torch.no_grad():
        for x, _, _, _, f in loader_te:
            ps, pm = model(x.to(DEVICE))
            s_all.append(ps.cpu().numpy()); m_all.append(pm.cpu().numpy()); f_all.append(f.numpy())
    return np.concatenate(f_all), np.concatenate(s_all), np.concatenate(m_all, axis=0)

def train_nn(model_cls, feats, stage_t, mech_t, folds, holdout_mask, label=''):
    n = len(feats)
    mu = np.nanmean(mech_t, axis=0); sd = np.nanstd(mech_t, axis=0); sd[sd < 1e-6] = 1.0
    mech_t_z = (mech_t - mu) / sd

    stage_pred = np.full(n, np.nan, dtype=np.float32)
    mech_pred  = np.full((n, 4), np.nan, dtype=np.float32)

    t0 = time.time()
    for fi, (tr_m, te_m) in enumerate(folds):
        tr_idx, te_idx = np.where(tr_m)[0], np.where(te_m)[0]
        f_, s_, m_ = train_one_split(model_cls, feats, stage_t, mech_t_z, tr_idx, te_idx)
        stage_pred[f_] = s_; mech_pred[f_] = m_ * sd + mu
        print(f'    fold {fi}: predicted {len(f_)} test frames')

    cv_idx, ho_idx = np.where(~holdout_mask)[0], np.where(holdout_mask)[0]
    f_, s_, m_ = train_one_split(model_cls, feats, stage_t, mech_t_z, cv_idx, ho_idx)
    full_stage = stage_pred.copy(); full_stage[f_] = s_
    full_mech  = mech_pred.copy();  full_mech [f_] = m_ * sd + mu

    cv_eval = np.zeros(n, dtype=bool)
    for _, te in folds: cv_eval |= te
    cv_eval &= np.isfinite(stage_pred)
    m_cv = compute_metrics(stage_pred[cv_eval], stage_t[cv_eval], mech_pred[cv_eval], mech_t[cv_eval])

    ho_eval = holdout_mask & np.isfinite(full_stage)
    m_ho = compute_metrics(full_stage[ho_eval], stage_t[ho_eval], full_mech[ho_eval], mech_t[ho_eval])
    print(f'  -> done in {time.time()-t0:.1f}s  '
          f'CV mae={m_cv["stage_mae"]:.3f} acc={m_cv["stage_acc"]:.3f}   '
          f'HO mae={m_ho["stage_mae"]:.3f} acc={m_ho["stage_acc"]:.3f}')
    return full_stage, full_mech, m_cv, m_ho

### Run CNN over every feature set

In [13]:
cnn_results = {}
for name, feats in FEATURE_SETS_NORM.items():
    print(f'\n[CNN] {name}')
    sp, mp, m_cv, m_ho = train_nn(TempCNN, feats, stage, aux_target, folds, holdout_mask, label=name)
    cnn_results[name] = dict(cv=m_cv, holdout=m_ho, stage_pred=sp, mech_pred=mp)


[CNN] LK-2D


    fold 0: predicted 1500 test frames


    fold 1: predicted 1500 test frames


    fold 2: predicted 1435 test frames


    fold 3: predicted 1450 test frames


...

    fold 1: predicted 1500 test frames


    fold 2: predicted 1435 test frames


    fold 3: predicted 1450 test frames


    fold 4: predicted 1450 test frames


  -> done in 46.5s  CV mae=0.033 acc=0.985   HO mae=0.523 acc=0.539


### Run Transformer over every feature set

In [14]:
tr_results = {}
for name, feats in FEATURE_SETS_NORM.items():
    print(f'\n[Transformer] {name}')
    sp, mp, m_cv, m_ho = train_nn(TempTransformer, feats, stage, aux_target, folds, holdout_mask, label=name)
    tr_results[name] = dict(cv=m_cv, holdout=m_ho, stage_pred=sp, mech_pred=mp)


[Transformer] LK-2D


    fold 0: predicted 1500 test frames


    fold 1: predicted 1500 test frames


    fold 2: predicted 1435 test frames


    fold 3: predicted 1450 test frames


...

    fold 1: predicted 1500 test frames


    fold 2: predicted 1435 test frames


    fold 3: predicted 1450 test frames


    fold 4: predicted 1450 test frames


  -> done in 65.9s  CV mae=0.023 acc=0.992   HO mae=0.539 acc=0.540


## 7. Aggregate results & metrics table

In [15]:
ALL_RESULTS = {'XGBoost': xgb_results, 'CNN': cnn_results, 'Transformer': tr_results}

rows = []
for model_name, by_feat in ALL_RESULTS.items():
    for feat_name, res in by_feat.items():
        for split in ('cv', 'holdout'):
            row = {'model': model_name, 'features': feat_name, 'split': split}
            row.update(res[split])
            rows.append(row)
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(OUT_DIR / 'metrics.csv', index=False)

print('── stage MAE (lower is better) ─────────────────────────────────────')
print(metrics_df.pivot_table(index='features', columns=['model','split'], values='stage_mae').round(3))
print('\n── stage accuracy (higher is better) ──────────────────────────────')
print(metrics_df.pivot_table(index='features', columns=['model','split'], values='stage_acc').round(3))
print('\n── onset error in seconds ─────────────────────────────────────────')
print(metrics_df.pivot_table(index='features', columns=['model','split'], values='onset_err_s').round(2))

── stage MAE (lower is better) ─────────────────────────────────────
model          CNN         Transformer         XGBoost        
split           cv holdout          cv holdout      cv holdout
features                                                      
LK+MS-3D-nn  0.033   0.523       0.023   0.539   0.004   0.150
LK-2D        0.027   0.211       0.029   0.358   0.008   0.314
LK-3D        0.021   0.325       0.014   0.176   0.004   0.150
MS-2D        0.110   2.849       0.115   3.025   0.219   3.016
MS-3D-nn     0.139   2.584       0.118   2.594   0.203   2.887
MS-3D-raw    0.148   2.943       0.131   3.103   0.228   2.999

── stage accuracy (higher is better) ──────────────────────────────
model          CNN         Transformer         XGBoost        
split           cv holdout          cv holdout      cv holdout
features                                                      
LK+MS-3D-nn  0.985   0.539       0.992   0.540   0.996   0.932
LK-2D        0.987   0.876       0.984   0.

## 8. Plots

### 8a. Predicted timelines per feature set

In [16]:
fig, axes = plt.subplots(len(FEATURE_SETS_NORM), 1,
                          figsize=(13, 1.9 * len(FEATURE_SETS_NORM)), sharex=True)
t = np.arange(n_frames) / FPS
ho_start_t = float(t[np.where(holdout_mask)[0].min()])
colors = {'XGBoost': 'tab:orange', 'CNN': 'tab:blue', 'Transformer': 'tab:green'}

for ax, (name, _) in zip(axes, FEATURE_SETS_NORM.items()):
    ax.plot(t, stage, '-', lw=1.8, color='k', alpha=0.75, label='truth')
    for model_name, c in colors.items():
        s_p = ALL_RESULTS[model_name][name]['stage_pred']
        ax.plot(t, np.clip(s_p, 0, len(STAGE_NAMES)-1), '-', lw=0.7, alpha=0.75, color=c, label=model_name)
    ax.axvspan(ho_start_t, t[-1], color='lightgray', alpha=0.35, lw=0)
    ax.set_yticks(range(len(STAGE_NAMES))); ax.set_yticklabels(STAGE_NAMES, fontsize=7)
    ax.set_ylabel(name, fontsize=9)
    ax.legend(fontsize=7, ncol=4, loc='upper left', frameon=False)
axes[-1].set_xlabel('time [s]')
fig.suptitle('Predicted stage timelines (gray strip = held-out final 10 %)', y=1.001, fontsize=11)
plt.tight_layout()
plt.savefig(OUT_DIR / 'predicted_timelines.png', dpi=120, bbox_inches='tight')
plt.show()

### 8b. Holdout MAE bar chart

In [17]:
fig, ax = plt.subplots(figsize=(12, 4.2))
feats_list  = list(FEATURE_SETS_NORM.keys())
models_list = ['XGBoost', 'CNN', 'Transformer']
x = np.arange(len(feats_list)); w = 0.27
for i, m in enumerate(models_list):
    vals = [ALL_RESULTS[m][f]['holdout']['stage_mae'] for f in feats_list]
    ax.bar(x + (i - 1) * w, vals, w, label=m, color=colors[m])
ax.set_xticks(x); ax.set_xticklabels(feats_list, rotation=15)
ax.set_ylabel('stage MAE (holdout)')
ax.set_title('Holdout stage MAE per feature set × model')
ax.grid(axis='y', alpha=0.3); ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'mae_bar.png', dpi=120, bbox_inches='tight')
plt.show()

### 8c. Onset timing table (CNN on holdout)

In [18]:
onset_rows = []
true_onsets = find_onsets(stage)
for model_name in models_list:
    for feat_name in feats_list:
        s_p = ALL_RESULTS[model_name][feat_name]['stage_pred']
        po = find_onsets(s_p)
        for k in range(1, len(STAGE_NAMES)):
            t_t = true_onsets[k] / FPS if true_onsets[k] is not None else None
            t_p = po[k] / FPS if po[k] is not None else None
            err = abs(t_p - t_t) if (t_t is not None and t_p is not None) else None
            onset_rows.append(dict(model=model_name, features=feat_name,
                                    stage=STAGE_NAMES[k],
                                    true_s=t_t, pred_s=t_p, err_s=err))
onset_df = pd.DataFrame(onset_rows)
onset_df.to_csv(OUT_DIR / 'onset_table.csv', index=False)

print('Onset timing error [s] — CNN:')
print(onset_df[onset_df.model == 'CNN']
      .pivot(index='features', columns='stage', values='err_s')[STAGE_NAMES[1:]].round(2))

Onset timing error [s] — CNN:
stage        lever1_pivoted  slider1_slid  ball_removed  cover_slid
features                                                           
LK+MS-3D-nn          147.40          0.87          0.33        0.97
LK-2D                  0.03          1.07          0.07        1.10
LK-3D                  0.23          0.60          0.10        1.10
MS-2D                162.73        169.43         26.03        0.00
MS-3D-nn             162.80        169.50          1.07        0.23
MS-3D-raw            162.83        169.57         28.20        0.10


## 9. Save everything

`predictions.pkl` carries the per-frame predictions for every (model × feature-set)
plus the truth arrays and CV folds, so downstream notebooks (ensembles, per-stage
confusion, paw-trajectory overlays) can pick up without re-training.

In [19]:
save = dict(
    folds=[(tr, te) for tr, te in folds],
    holdout_mask=holdout_mask,
    stage_true=stage, aux_target=aux_target,
    STAGE_NAMES=STAGE_NAMES, STAGE_ORDER=STAGE_ORDER, FPS=FPS,
    feature_dims={k: v.shape[1] for k, v in FEATURE_SETS_NORM.items()},
    results={m: {f: {'cv': r['cv'], 'holdout': r['holdout'],
                     'stage_pred': r['stage_pred'], 'mech_pred': r['mech_pred']}
                 for f, r in by_feat.items()}
             for m, by_feat in ALL_RESULTS.items()},
)
with open(OUT_DIR / 'predictions.pkl', 'wb') as f:
    pickle.dump(save, f)
print(f'Saved -> {OUT_DIR}')
print('   metrics.csv, onset_table.csv, predictions.pkl, predicted_timelines.png, mae_bar.png')

Saved -> /home/kenny/HTCV/data/lockbox_state_prediction/scene1
   metrics.csv, onset_table.csv, predictions.pkl, predicted_timelines.png, mae_bar.png
